<a href="https://colab.research.google.com/github/DzhuJK/ML/blob/main/LR_7_Rec_Sist_%D0%9B%D0%B0%D1%86_4_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install scikit-surprise

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 44.6 MB/s  0:00:00


1. Доцільна реалізація

In [2]:
from surprise import Dataset, Reader
from surprise import SVD, KNNBasic
from surprise.model_selection import cross_validate, GridSearchCV
from surprise import accuracy
from surprise.model_selection import train_test_split

import sys
import io

In [3]:
!wget --no-check-certificate https://files.grouplens.org/datasets/movielens/ml-100k.zip

!unzip -o ml-100k.zip

--2026-11-11 12:27:21--  https://files.grouplens.org/datasets/movielens/ml-100k.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4924029 (4.7M) [application/zip]
Saving to: ‘ml-100k.zip’

ml-100k.zip         100%[===================>]   4.70M  2.80MB/s    in 1.7s    

2026-11-11 12:27:24 (2.80 MB/s) - ‘ml-100k.zip’ saved [4924029/4924029]

Archive:  ml-100k.zip
   creating: ml-100k/
  inflating: ml-100k/allbut.pl       
  inflating: ml-100k/mku.sh          
  inflating: ml-100k/README          
  inflating: ml-100k/u.data          
  inflating: ml-100k/u.genre         
  inflating: ml-100k/u.info          
  inflating: ml-100k/u.item          
  inflating: ml-100k/u.occupation    
  inflating: ml-100k/u.user          
  inflating: ml-100k/u1.base         
  inflating: ml-100k/u1.test         
  inflating: ml-100k/u2.ba

In [4]:
import pandas as pd

data = pd.read_csv('ml-100k/u.data', sep='\t', names=['user_id', 'movie_id', 'rating', 'timestamp'])
data.head()

,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [5]:
reader = Reader(rating_scale=(1, 5))
data_surprise = Dataset.load_from_df(data[['user_id', 'movie_id', 'rating']], reader)

In [6]:
trainset, testset = train_test_split(data_surprise, test_size=0.2)

# 3. GridSearchCV для SVD
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

grid_search_svd = GridSearchCV(SVD, param_grid_svd, measures=['rmse', 'mae'], cv=3)
grid_search_svd.fit(data_surprise)

print("Best RMSE for SVD:", grid_search_svd.best_score['rmse'])
print("Best params for SVD:", grid_search_svd.best_params['rmse'])

# 4. GridSearchCV для KNNBasic
param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [True, False]
    }
}

# Приглушення виводу similarity matrix
sys_stdout_backup = sys.stdout
sys.stdout = io.StringIO()

grid_search_knn = GridSearchCV(KNNBasic, param_grid_knn, measures=['rmse', 'mae'], cv=3)
grid_search_knn.fit(data_surprise)

# Повернення нормального виводу
sys.stdout = sys_stdout_backup

print("Best RMSE for KNNBasic:", grid_search_knn.best_score['rmse'])
print("Best params for KNNBasic:", grid_search_knn.best_params['rmse'])

# 5. Найкращий SVD
best_svd = SVD(**grid_search_svd.best_params['rmse'])
best_svd.fit(trainset)
predictions_svd = best_svd.test(testset)
print("Test RMSE (SVD):", accuracy.rmse(predictions_svd))

# 6. Найкращий KNNBasic
best_knn = KNNBasic(**grid_search_knn.best_params['rmse'])
best_knn.fit(trainset)
predictions_knn = best_knn.test(testset)
print("Test RMSE (KNNBasic):", accuracy.rmse(predictions_knn))

Best RMSE for SVD: 0.9313577721912258
Best params for SVD: {'n_factors': 100, 'lr_all': 0.01, 'reg_all': 0.05}
Best RMSE for KNNBasic: 1.0180724209920946
Best params for KNNBasic: {'k': 40, 'min_k': 5, 'sim_options': {'name': 'pearson', 'user_based': True}}
RMSE: 0.9181
Test RMSE (SVD): 0.9181008187736286
Computing the pearson similarity matrix...
Done computing similarity matrix.
RMSE: 1.0139
Test RMSE (KNNBasic): 1.0138931422287327


In [7]:
best_svd = grid_search_svd.best_estimator['mae']
best_knn = grid_search_knn.best_estimator['mae']

print(f"Найкращий MAE для SVD: {grid_search_svd.best_score['mae']}")
print(f"Найкращий MAE для KNNBasic: {grid_search_knn.best_score['mae']}")

# Вибір моделі з найменшим MAE
if grid_search_svd.best_score['mae'] < grid_search_knn.best_score['mae']:
    best_model = best_svd
    print("Вибраний алгоритм: SVD")
else:
    best_model = best_knn
    print("Вибраний алгоритм: KNNBasic")



Найкращий MAE для SVD: 0.734193847297425
Найкращий MAE для KNNBasic: 0.8078274387887093
Вибраний алгоритм: SVD


In [8]:
for i, (uid, iid, rating) in enumerate(trainset.all_ratings()):
    print(uid, iid, rating)
    if i > 10:
        break

0 0 4.0
0 39 3.0
0 205 4.0
0 3 3.0
0 523 2.0
0 651 1.0
0 86 4.0
0 268 5.0
0 2 5.0
0 533 3.0
0 47 5.0
0 42 2.0


In [13]:
best_model.fit(trainset)

# Вибираємо реального користувача, який ГАРАНТОВАНО є в trainset
# Беремо перший внутрішній ID з тренувального набору
inner_uid = list(trainset.ur.keys())[0]
user_id = trainset.to_raw_uid(inner_uid)  # отримуємо реальний ID

# Оцінки цього користувача в trainset
user_ratings = trainset.ur[inner_uid]
print(f"Кількість рецензій користувача {user_id}: {len(user_ratings)}")

# Усі фільми та вже оцінені фільми
all_items = set(trainset.all_items())
rated_items = set(item for item, _ in user_ratings)
unrated_items = all_items - rated_items

# Прогнозування рейтингів для неоцінених фільмів
predictions = []
for item in unrated_items:
    raw_item_id = trainset.to_raw_iid(item)
    est_rating = best_model.predict(user_id, raw_item_id).est
    predictions.append((raw_item_id, est_rating))

predictions.sort(key=lambda x: x[1], reverse=True)

# Виведення 10 найкращих фільмів
print("Топ-10 фільмів, рекомендованих для користувача:")
for item_id, rating in predictions[:10]:
    print(f"Фільм {item_id} з прогнозованим рейтингом {rating:.2f}")

Кількість рецензій користувача 648: 244
Топ-10 фільмів, рекомендованих для користувача:
Фільм 313 з прогнозованим рейтингом 4.60
Фільм 50 з прогнозованим рейтингом 4.54
Фільм 270 з прогнозованим рейтингом 4.36
Фільм 12 з прогнозованим рейтингом 4.34
Фільм 199 з прогнозованим рейтингом 4.32
Фільм 64 з прогнозованим рейтингом 4.29
Фільм 408 з прогнозованим рейтингом 4.28
Фільм 705 з прогнозованим рейтингом 4.27
Фільм 170 з прогнозованим рейтингом 4.25
Фільм 166 з прогнозованим рейтингом 4.25


In [16]:
best_model.fit(trainset)

# Беремо користувача, який ГАРАНТОВАНО є в trainset
inner_uid = list(trainset.ur.keys())[0]
user_id = trainset.to_raw_uid(inner_uid)

user_ratings = trainset.ur[inner_uid]
print(f"Кількість рецензій користувача {user_id}: {len(user_ratings)}")

# Отримання рекомендацій
all_items = set(trainset.all_items())
rated_items = set([item for (item, _) in user_ratings])
unrated_items = all_items - rated_items

# Прогнозування рейтингів (зберігаємо реальний ID фільму одразу)
predictions = []
for item in unrated_items:
    raw_item = trainset.to_raw_iid(item)
    predictions.append((raw_item, best_model.predict(user_id, raw_item).est))

predictions.sort(key=lambda x: x[1], reverse=True)

# Виведення 10 найкращих фільмів
print("Топ-10 фільмів, рекомендованих для користувача:")
for item_id, rating in predictions[:10]:
    print(f"Фільм {item_id} з прогнозованим рейтингом {rating:.2f}")

Кількість рецензій користувача 648: 244
Топ-10 фільмів, рекомендованих для користувача:
Фільм 300 з прогнозованим рейтингом 4.78
Фільм 313 з прогнозованим рейтингом 4.71
Фільм 50 з прогнозованим рейтингом 4.57
Фільм 641 з прогнозованим рейтингом 4.56
Фільм 64 з прогнозованим рейтингом 4.52
Фільм 520 з прогнозованим рейтингом 4.47
Фільм 328 з прогнозованим рейтингом 4.39
Фільм 199 з прогнозованим рейтингом 4.37
Фільм 125 з прогнозованим рейтингом 4.35
Фільм 195 з прогнозованим рейтингом 4.34
